# 🏥 CT DICOM — Images & Metadata
**Dataset:** [kmader/siim-medical-images](https://www.kaggle.com/datasets/kmader/siim-medical-images)  
Reads pixel data **and** clinical metadata directly from the `.dcm` files in `dicom_dir/`.

---
## 1️⃣  Install & Import

In [ ]:
# !pip install pydicom matplotlib numpy pandas scipy pillow seaborn

import os, glob, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from scipy.ndimage import sobel
from PIL import Image
import io

import pydicom
from pydicom import dcmread

print(f'pydicom    : {pydicom.__version__}')
import matplotlib; print(f'matplotlib : {matplotlib.__version__}')
print(f'numpy      : {np.__version__}')
print(f'pandas     : {pd.__version__}')

BG, PANEL, BORDER = '#0d1117', '#161b22', '#30363d'
BLUE, GREEN, PINK, YELLOW = '#38bdf8', '#34d399', '#fb7185', '#fbbf24'
LABEL, TITLE = '#94a3b8', '#7dd3fc'

---
## 2️⃣  Configure Path

In [ ]:
# ── Point this to your downloaded folder ─────────────────────────────────────
DATA_DIR  = r'.\siim-medical-images'               # Windows
# DATA_DIR = './siim-medical-images'               # Mac / Linux
# DATA_DIR = '/kaggle/input/siim-medical-images'   # Kaggle notebook
# ─────────────────────────────────────────────────────────────────────────────

DICOM_DIR = os.path.join(DATA_DIR, 'dicom_dir')

# Find all .dcm files — handles both flat and nested layouts
dcm_files = sorted(glob.glob(os.path.join(DICOM_DIR, '**', '*.dcm'), recursive=True))
if not dcm_files:
    dcm_files = sorted(glob.glob(os.path.join(DICOM_DIR, '*.dcm')))

print(f'✓ Found {len(dcm_files)} DICOM files in {DICOM_DIR}')
if dcm_files:
    print(f'  First: {os.path.basename(dcm_files[0])}')
    print(f'  Last : {os.path.basename(dcm_files[-1])}')

---
## 3️⃣  Read One DICOM File — All Tags

In [ ]:
# Load the first file and print every tag it contains
ds = dcmread(dcm_files[0])

print('=' * 60)
print('  ALL DICOM TAGS IN THIS FILE')
print('=' * 60)
for elem in ds:
    if elem.keyword == 'PixelData':
        # Skip the raw binary blob — too large to display
        print(f'  {elem.keyword:<42} <pixel data blob>')
    else:
        print(f'  {elem.keyword:<42} {elem.value}')

---
## 4️⃣  Extract Pixel Data → Hounsfield Units

```
HU = raw_pixel × RescaleSlope + RescaleIntercept
```

| Tissue | HU |
|---|---|
| Air | −1000 |
| Lung | −600 → −400 |
| Fat | −100 → −50 |
| Water | 0 |
| Brain | 20 → 80 |
| Blood | 30 → 45 |
| Bone | 400 → 1000 |

In [ ]:
def to_hu(ds):
    """Convert raw DICOM pixel array to Hounsfield Units."""
    px    = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, 'RescaleSlope',     1.0))
    inter = float(getattr(ds, 'RescaleIntercept', 0.0))
    return px * slope + inter


def apply_window(hu, wc, ww):
    """Apply a clinical display window → uint8 [0, 255]."""
    lo, hi = wc - ww / 2, wc + ww / 2
    return ((np.clip(hu, lo, hi) - lo) / (hi - lo) * 255).astype(np.uint8)


hu = to_hu(ds)
print(f'Shape         : {hu.shape}')
print(f'dtype         : {hu.dtype}')
print(f'HU range      : [{hu.min():.0f},  {hu.max():.0f}]')
print(f'Unique values : {len(np.unique(hu))}  '
      f'(vs max 256 in an 8-bit JPEG/PNG)')

---
## 5️⃣  Batch-Read All Files → Metadata Table

In [ ]:
# Tags to extract from every file
META_TAGS = [
    'PatientID', 'PatientName', 'PatientAge', 'PatientSex',
    'StudyDate', 'Modality', 'StudyDescription',
    'Rows', 'Columns', 'BitsAllocated',
    'SliceThickness', 'PixelSpacing',
    'KVP', 'XRayTubeCurrent',
    'RescaleSlope', 'RescaleIntercept',
    'SOPInstanceUID',
]


def extract_metadata(path):
    """Read tags from one .dcm file (skips pixel data for speed)."""
    d   = dcmread(path, stop_before_pixels=True)
    row = {'file': os.path.basename(path)}
    for tag in META_TAGS:
        val = getattr(d, tag, None)
        # Flatten multi-value types (DSfloat, IS, sequences) to plain Python
        if hasattr(val, '__iter__') and not isinstance(val, str):
            val = list(val)
        row[tag] = val
    return row


def parse_age(a):
    """Convert DICOM age string '045Y' to integer 45."""
    try:
        return int(str(a).upper().replace('Y', '').strip())
    except Exception:
        return np.nan


print(f'Reading metadata from {len(dcm_files)} files…')
records  = [extract_metadata(p) for p in dcm_files]
meta_df  = pd.DataFrame(records)
meta_df['Age'] = meta_df['PatientAge'].apply(parse_age)

print(f'✓ Metadata table: {meta_df.shape[0]} rows × {meta_df.shape[1]} columns')
meta_df[['file','PatientID','Age','PatientSex','Modality',
         'StudyDate','BitsAllocated','SliceThickness']].head(10)

---
## 6️⃣  Metadata Summary

In [ ]:
print('── Numeric columns ─────────────────────────────────────────')
num_cols = [c for c in ['Age','Rows','Columns','BitsAllocated',
                        'RescaleSlope','RescaleIntercept','KVP']
            if c in meta_df.columns]
print(pd.to_numeric(meta_df[num_cols].stack(), errors='coerce')
        .unstack().describe().round(2).to_string())

print('\n── Categorical columns ──────────────────────────────────────')
for col in ['Modality', 'PatientSex', 'StudyDescription']:
    if col in meta_df.columns:
        print(f'\n{col}:')
        print(meta_df[col].value_counts().to_string())

---
## 7️⃣  Metadata Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor(BG)
fig.suptitle('DICOM Metadata — Dataset Overview',
             color='white', fontsize=13, fontweight='bold')

def style(ax, title, xlabel, ylabel='Count'):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=TITLE, fontsize=11)
    ax.set_xlabel(xlabel, color=LABEL)
    ax.set_ylabel(ylabel, color=LABEL)
    ax.tick_params(colors=LABEL)
    for s in ax.spines.values(): s.set_edgecolor(BORDER)

# Age distribution
ages = pd.to_numeric(meta_df.get('Age'), errors='coerce').dropna()
if len(ages):
    axes[0].hist(ages, bins=20, color=BLUE, edgecolor=BG)
    axes[0].axvline(ages.mean(), color=YELLOW, lw=1.5, ls='--',
                    label=f'Mean {ages.mean():.0f} y')
    axes[0].legend(labelcolor='white', facecolor='#21262d',
                   edgecolor=BORDER, fontsize=9)
else:
    axes[0].text(0.5, 0.5, 'Age not available', ha='center',
                 va='center', color=LABEL, transform=axes[0].transAxes)
style(axes[0], 'Patient Age Distribution', 'Age (years)')

# Modality bar chart
if 'Modality' in meta_df.columns:
    counts = meta_df['Modality'].value_counts()
    bars   = axes[1].bar(counts.index, counts.values,
                         color=[GREEN, BLUE, PINK, YELLOW][:len(counts)],
                         edgecolor=BG)
    for bar, v in zip(bars, counts.values):
        axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.3,
                     str(v), ha='center', color='white', fontsize=9)
style(axes[1], 'Modality', 'Modality')

# Patient sex pie
if 'PatientSex' in meta_df.columns:
    counts = meta_df['PatientSex'].fillna('Unknown').value_counts()
    axes[2].pie(counts.values, labels=counts.index, autopct='%1.0f%%',
                colors=[BLUE, PINK, YELLOW][:len(counts)],
                textprops={'color': 'white'},
                wedgeprops={'edgecolor': BG, 'linewidth': 2})
style(axes[2], 'Patient Sex', '', '')

plt.tight_layout()
plt.show()

---
## 8️⃣  DICOM vs JPEG/PNG — Why Format Matters

| Feature | DICOM `.dcm` | JPEG / PNG |
|---|---|---|
| **Bit depth** | 12–16 bit (up to 65 536 gray levels) | 8 bit (256 levels) |
| **Pixel values** | Hounsfield Units — absolute tissue density | sRGB display values |
| **Metadata** | Full clinical record embedded in every file | Minimal EXIF only |
| **Compression** | Lossless or uncompressed | JPEG is lossy |
| **Multi-frame** | One file can hold an entire CT/MRI series | One file per image |
| **Standard** | ISO 12052 — used by all medical equipment | General web format |

---
## 9️⃣  Clinical Windowing — Same File, Six Views

In [ ]:
WINDOWS = {
    'Brain'    : ( 40,   80),
    'Subdural' : ( 75,  215),
    'Stroke'   : ( 32,   48),
    'Bone'     : (400, 1800),
    'Lung'     : (-600,1500),
    'Abdomen'  : ( 60,  400),
}

hu_sample = to_hu(dcmread(dcm_files[0]))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.patch.set_facecolor(BG)
fig.suptitle('CT Windowing — Same DICOM file, six clinical presets',
             color='white', fontsize=14, fontweight='bold', y=1.01)

for ax, (name, (wc, ww)) in zip(axes.flat, WINDOWS.items()):
    ax.set_facecolor(PANEL)
    ax.imshow(apply_window(hu_sample, wc, ww), cmap='gray')
    ax.set_title(f'{name}\nWC={wc}  WW={ww}', color=TITLE, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 🔟  HU Histogram: 16-bit vs Simulated 8-bit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('Pixel Range — DICOM (16-bit) vs Simulated 8-bit JPEG',
             color='white', fontsize=13, fontweight='bold')

HU_REF = [(-1000,'Air','#94a3b8'),(-50,'Fat','#a78bfa'),
           (0,'Water','#60a5fa'),(40,'Brain','#f9a8d4'),(400,'Bone',YELLOW)]

for ax in axes:
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=LABEL)
    for s in ax.spines.values(): s.set_edgecolor(BORDER)

flat = hu_sample.flatten()
axes[0].hist(flat, bins=200, color=BLUE, edgecolor='none', alpha=0.85)
ymax = axes[0].get_ylim()[1]
for hv, lab, col in HU_REF:
    if flat.min() <= hv <= flat.max():
        axes[0].axvline(hv, color=col, lw=1.2, ls='--')
        axes[0].text(hv+20, ymax*0.55, lab, color=col, fontsize=8,
                     rotation=90, va='top')
axes[0].set_title(f'DICOM — {len(np.unique(flat))} unique HU values', color=TITLE)
axes[0].set_xlabel('HU Value', color=LABEL)
axes[0].set_ylabel('Pixel Count', color=LABEL)

bw = apply_window(hu_sample, 40, 400)
pct = (1 - len(np.unique(bw)) / len(np.unique(flat))) * 100
axes[1].hist(bw.flatten(), bins=64, color=PINK, edgecolor='none', alpha=0.85)
axes[1].set_title(
    f'Simulated 8-bit — {len(np.unique(bw))} unique values\n'
    f'{pct:.0f}% of HU range discarded', color=TITLE)
axes[1].set_xlabel('Value (0–255)', color=LABEL)
axes[1].set_ylabel('Pixel Count', color=LABEL)

for ax, txt in [
    (axes[0], f'16-bit DICOM\n{len(np.unique(flat))} unique values'),
    (axes[1], f'8-bit JPEG/PNG\n{len(np.unique(bw))} unique values'),
]:
    ax.text(0.97, 0.97, txt, transform=ax.transAxes, fontsize=9,
            va='top', ha='right', color='white',
            bbox=dict(boxstyle='round,pad=0.4', fc='#21262d', ec=BORDER))

plt.tight_layout()
plt.show()

---
## 1️⃣1️⃣  Tissue Segmentation by Hounsfield Unit

Each tissue class is defined by a clinically established HU range:

| Label | HU range | Colour |
|---|---|---|
| Air | < −950 | Dark grey |
| Lung | −950 → −500 | Sky blue |
| Fat | −500 → −100 | Gold |
| Soft Tissue | −100 → 200 | Salmon |
| Blood / Contrast | 200 → 300 | Red |
| Bone | > 300 | White |

In [ ]:
# ── Tissue class definitions ──────────────────────────────────────────────────
# Each entry: (label, hu_min, hu_max, RGB colour 0-255)
TISSUES = [
    ('Air',              -np.inf,  -950, (30,  30,  30 )),
    ('Lung',               -950,   -500, (80,  180, 230)),
    ('Fat',                -500,   -100, (230, 185,  80)),
    ('Soft Tissue',        -100,    200, (220, 120, 100)),
    ('Blood / Contrast',    200,    300, (200,  50,  50)),
    ('Bone',                300,  np.inf,(240, 240, 240)),
]


def segment_tissues(hu_array, tissues=TISSUES):
    """
    Assign each pixel to a tissue class based on its HU value.

    Returns
    -------
    label_map : (H, W) int array  — index into `tissues` list (-1 = unclassified)
    rgb_map   : (H, W, 3) uint8   — colour-coded overlay
    """
    label_map = np.full(hu_array.shape, -1, dtype=np.int8)
    rgb_map   = np.zeros((*hu_array.shape, 3), dtype=np.uint8)

    for i, (name, lo, hi, colour) in enumerate(tissues):
        mask = (hu_array >= lo) & (hu_array < hi)
        label_map[mask] = i
        rgb_map[mask]   = colour

    return label_map, rgb_map


def tissue_stats(hu_array, label_map, tissues=TISSUES, pixel_area_mm2=None):
    """
    Compute per-tissue pixel count, percentage, mean HU, std HU.
    If pixel_area_mm2 is given, also compute approximate area in cm².
    """
    total = hu_array.size
    rows  = []
    for i, (name, lo, hi, colour) in enumerate(tissues):
        mask   = label_map == i
        count  = int(mask.sum())
        if count == 0:
            rows.append({'Tissue': name, 'Pixels': 0, '%': 0.0,
                         'Mean HU': np.nan, 'Std HU': np.nan,
                         'Area cm²': 0.0})
            continue
        vals = hu_array[mask]
        area = (count * pixel_area_mm2 / 100) if pixel_area_mm2 else np.nan
        rows.append({
            'Tissue'   : name,
            'Pixels'   : count,
            '%'        : round(count / total * 100, 2),
            'Mean HU'  : round(float(vals.mean()), 1),
            'Std HU'   : round(float(vals.std()),  1),
            'Area cm²' : round(area, 2) if not np.isnan(area) else np.nan,
        })
    return pd.DataFrame(rows)


# ── Run segmentation on the first slice ───────────────────────────────────────
ds0       = dcmread(dcm_files[0])
hu0       = to_hu(ds0)

# Get pixel spacing for area calculation (mm → cm²)
ps        = getattr(ds0, 'PixelSpacing', None)
px_area   = float(ps[0]) * float(ps[1]) if ps else None

label_map, rgb_map = segment_tissues(hu0)
stats_tbl          = tissue_stats(hu0, label_map, pixel_area_mm2=px_area)

print('Tissue segmentation — per-class statistics')
print(stats_tbl.to_string(index=False))

In [ ]:
# ── Figure 1: CT image | Segmentation mask | Overlay ─────────────────────────
abdomen_win = apply_window(hu0, 60, 400)

# Semi-transparent overlay: blend greyscale CT with colour mask
ct_rgb   = np.stack([abdomen_win]*3, axis=-1)          # H×W×3 greyscale
overlay  = (ct_rgb * 0.45 + rgb_map * 0.55).clip(0,255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('Tissue Segmentation by Hounsfield Unit',
             color='white', fontsize=14, fontweight='bold')

titles  = ['Original CT (Abdomen Window)', 'Tissue Mask', 'Overlay (45% CT + 55% mask)']
imgs    = [abdomen_win, rgb_map, overlay]
cmaps   = ['gray', None, None]

for ax, title, img, cmap in zip(axes, titles, imgs, cmaps):
    ax.set_facecolor(PANEL)
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, color=TITLE, fontsize=11)
    ax.axis('off')

# Shared legend
patches = [
    mpatches.Patch(color=np.array(c)/255, label=name)
    for name, *_, c in TISSUES
]
axes[1].legend(handles=patches, loc='lower right', fontsize=8,
               labelcolor='white', facecolor='#21262d', edgecolor=BORDER)

plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: Per-tissue HU distributions ─────────────────────────────────────
present = [(i, name, lo, hi, col)
           for i, (name, lo, hi, col) in enumerate(TISSUES)
           if (label_map == i).sum() > 0]

n_tissues = len(present)
cols_g    = 3
rows_g    = int(np.ceil(n_tissues / cols_g))

fig, axes = plt.subplots(rows_g, cols_g,
                          figsize=(cols_g * 5, rows_g * 3.5))
fig.patch.set_facecolor(BG)
fig.suptitle('HU Distribution per Tissue Class',
             color='white', fontsize=13, fontweight='bold')

for ax, (i, name, lo, hi, col) in zip(axes.flat, present):
    ax.set_facecolor(PANEL)
    vals = hu0[label_map == i]
    colour_norm = np.array(col) / 255
    ax.hist(vals, bins=80, color=colour_norm, edgecolor='none', alpha=0.9)

    mean_v, std_v = vals.mean(), vals.std()
    ax.axvline(mean_v, color='white', lw=1.2, ls='--',
               label=f'mean {mean_v:.0f} HU')
    ax.axvspan(mean_v - std_v, mean_v + std_v,
               alpha=0.15, color='white', label=f'±1σ ({std_v:.0f})')

    ax.set_title(name, color=TITLE, fontsize=10)
    ax.set_xlabel('HU', color=LABEL, fontsize=8)
    ax.set_ylabel('Pixel count', color=LABEL, fontsize=8)
    ax.tick_params(colors=LABEL, labelsize=7)
    ax.legend(fontsize=7, labelcolor='white',
              facecolor='#21262d', edgecolor=BORDER)
    for s in ax.spines.values(): s.set_edgecolor(BORDER)

    row = stats_tbl[stats_tbl.Tissue == name].iloc[0]
    ax.text(0.97, 0.97,
            f"{row['%']:.1f}% of slice\n{row['Pixels']:,} px",
            transform=ax.transAxes, fontsize=8, va='top', ha='right',
            color='white',
            bbox=dict(boxstyle='round,pad=0.3', fc='#21262d', ec=BORDER))

for ax in axes.flat[n_tissues:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 3: Pixel-area breakdown bar + pie ──────────────────────────────────
df_plot = stats_tbl[stats_tbl['Pixels'] > 0].copy()
colours = [np.array(c)/255 for _, _, _, c in
           [(t[0], t[1], t[2], t[3]) for t in TISSUES
            if t[0] in df_plot['Tissue'].values]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('Tissue Area Breakdown', color='white',
             fontsize=13, fontweight='bold')

# Bar chart — % of slice
ax = axes[0]
ax.set_facecolor(PANEL)
bars = ax.bar(df_plot['Tissue'], df_plot['%'], color=colours, edgecolor=BG)
for bar, pct in zip(bars, df_plot['%']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{pct:.1f}%', ha='center', color='white', fontsize=9)
ax.set_title('% of Slice per Tissue', color=TITLE)
ax.set_ylabel('% of pixels', color=LABEL)
ax.tick_params(colors=LABEL, axis='both')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right', color=LABEL)
for s in ax.spines.values(): s.set_edgecolor(BORDER)

# Pie chart
ax2 = axes[1]
ax2.set_facecolor(PANEL)
wedges, texts, autotexts = ax2.pie(
    df_plot['%'], labels=df_plot['Tissue'],
    autopct='%1.1f%%', colors=colours,
    textprops={'color': 'white', 'fontsize': 9},
    wedgeprops={'edgecolor': BG, 'linewidth': 1.5},
    startangle=140
)
for t in autotexts: t.set_fontsize(8)
ax2.set_title('Tissue Composition', color=TITLE)

plt.tight_layout()
plt.show()

print('\nFull statistics table:')
print(stats_tbl.to_string(index=False))

In [ ]:
# ── Figure 4: Segmentation across multiple slices ─────────────────────────────
n_show  = min(8, len(dcm_files))
fig, axes = plt.subplots(2, n_show, figsize=(n_show * 2.8, 6))
fig.patch.set_facecolor(BG)
fig.suptitle(f'Tissue Segmentation — {n_show} Slices',
             color='white', fontsize=13, fontweight='bold')

for col, path in enumerate(dcm_files[:n_show]):
    d   = dcmread(path)
    h   = to_hu(d)
    lm, rm = segment_tissues(h)
    win = apply_window(h, 60, 400)

    # Top row: CT
    axes[0, col].set_facecolor(PANEL)
    axes[0, col].imshow(win, cmap='gray')
    axes[0, col].set_title(f'Slice {col+1}', color=TITLE, fontsize=8)
    axes[0, col].axis('off')

    # Bottom row: mask
    axes[1, col].set_facecolor(PANEL)
    axes[1, col].imshow(rm)
    axes[1, col].axis('off')

# Row labels
axes[0, 0].set_ylabel('CT Image', color=LABEL, fontsize=9)
axes[1, 0].set_ylabel('Tissue Mask', color=LABEL, fontsize=9)

# Global legend on last column
patches = [mpatches.Patch(color=np.array(c)/255, label=n)
           for n, *_, c in TISSUES]
axes[1, -1].legend(handles=patches, loc='lower right', fontsize=7,
                   labelcolor='white', facecolor='#21262d', edgecolor=BORDER)

plt.tight_layout()
plt.show()

---
## 1️⃣2️⃣  Full Dashboard — One DICOM File

In [ ]:
ds0 = dcmread(dcm_files[0])
hu0 = to_hu(ds0)

fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor(BG)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.35)

# Row 0 — four windowed views
for spec, img, cmap, title in [
    (gs[0,0], apply_window(hu0,  40,   80), 'gray',      'Brain  WC=40  WW=80'),
    (gs[0,1], apply_window(hu0,  40,   80), 'hot',       'Brain — Hot colormap'),
    (gs[0,2], apply_window(hu0, 400, 1800), 'bone',      'Bone   WC=400 WW=1800'),
    (gs[0,3], apply_window(hu0,-600, 1500), 'gist_gray', 'Lung   WC=-600 WW=1500'),
]:
    ax = fig.add_subplot(spec)
    ax.set_facecolor(PANEL)
    im = ax.imshow(img, cmap=cmap)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(
        colors=LABEL, labelsize=7)
    ax.set_title(title, color=TITLE, fontsize=9, pad=5)
    ax.axis('off')

# Row 1a — Sobel edge map
ax_e = fig.add_subplot(gs[1,0])
ax_e.set_facecolor(PANEL)
bw = apply_window(hu0, 40, 80).astype(float)
ax_e.imshow(np.hypot(sobel(bw, 0), sobel(bw, 1)), cmap='inferno')
ax_e.set_title('Sobel Edge Map', color=TITLE, fontsize=9)
ax_e.axis('off')

# Row 1b — HU intensity profile
ax_p = fig.add_subplot(gs[1,1])
ax_p.set_facecolor(PANEL)
mid = hu0[hu0.shape[0]//2, :]
ax_p.plot(mid, color=BLUE, lw=1.2)
for hv, lab, col in [(-1000,'Air','#94a3b8'),(0,'Water','#60a5fa'),(400,'Bone',YELLOW)]:
    ax_p.axhline(hv, color=col, lw=0.8, ls='--', label=f'{lab} {hv} HU')
ax_p.set_title('HU Profile — Centre Row', color=TITLE, fontsize=9)
ax_p.set_xlabel('Column', color=LABEL, fontsize=7)
ax_p.set_ylabel('HU', color=LABEL, fontsize=7)
ax_p.tick_params(colors=LABEL, labelsize=7)
ax_p.legend(fontsize=7, labelcolor='white', facecolor='#21262d', edgecolor=BORDER)
for s in ax_p.spines.values(): s.set_edgecolor(BORDER)

# Row 1c — tissue segmentation (uses segment_tissues() from Section 11)
ax_s = fig.add_subplot(gs[1,2])
ax_s.set_facecolor(PANEL)
_, rgb_dash = segment_tissues(hu0)
ax_s.imshow(rgb_dash)
ax_s.legend(handles=[
    mpatches.Patch(color=np.array(c)/255, label=n)
    for n, lo, hi, c in TISSUES
], loc='lower right', fontsize=7, labelcolor='white',
   facecolor='#21262d', edgecolor=BORDER)
ax_s.set_title('Tissue Segmentation (6 classes)', color=TITLE, fontsize=9)
ax_s.axis('off')

# Row 1d — DICOM metadata panel
ax_i = fig.add_subplot(gs[1,3])
ax_i.set_facecolor(PANEL)
ax_i.axis('off')

def tag(d, name, default='N/A'):
    return getattr(d, name, default)

ps = tag(ds0, 'PixelSpacing', ['?', '?'])
info = (
    f"DICOM METADATA\n{'─'*28}\n"
    f"PatientID     {tag(ds0,'PatientID')}\n"
    f"Age           {tag(ds0,'PatientAge')}\n"
    f"Sex           {tag(ds0,'PatientSex')}\n"
    f"StudyDate     {tag(ds0,'StudyDate')}\n"
    f"Modality      {tag(ds0,'Modality')}\n"
    f"Description   {str(tag(ds0,'StudyDescription'))[:20]}\n"
    f"{'─'*28}\n"
    f"Shape         {hu0.shape[0]} x {hu0.shape[1]}\n"
    f"Bits alloc    {tag(ds0,'BitsAllocated')}\n"
    f"Pixel spacing {ps[0]} mm\n"
    f"Slice thick   {tag(ds0,'SliceThickness')} mm\n"
    f"KVP           {tag(ds0,'KVP')} kV\n"
    f"{'─'*28}\n"
    f"HU min        {hu0.min():.0f}\n"
    f"HU max        {hu0.max():.0f}\n"
    f"HU mean       {hu0.mean():.1f}\n"
    f"Unique vals   {len(np.unique(hu0))}"
)
ax_i.text(0.05, 0.97, info, transform=ax_i.transAxes,
          fontsize=8.5, va='top', color='#e2e8f0', family='monospace',
          bbox=dict(boxstyle='round,pad=0.6', fc='#21262d', ec=BORDER))
ax_i.set_title('DICOM Tags', color=TITLE, fontsize=9)

fig.suptitle('DICOM Analysis Dashboard — CT Medical Images',
             color='white', fontsize=13, fontweight='bold', y=1.01)
plt.show()

---
## 1️⃣2️⃣  Browse 12 Files — Image + Metadata per Panel

In [ ]:
n_show = min(12, len(dcm_files))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.patch.set_facecolor(BG)
fig.suptitle(f'First {n_show} DICOM Slices — Abdomen Window + Key Tags',
             color='white', fontsize=13, fontweight='bold')

for ax, path in zip(axes.flat, dcm_files[:n_show]):
    d  = dcmread(path)
    h  = to_hu(d)
    ax.set_facecolor(PANEL)
    ax.imshow(apply_window(h, 60, 400), cmap='gray')
    pid = getattr(d, 'PatientID',  '?')
    age = getattr(d, 'PatientAge', '?')
    sex = getattr(d, 'PatientSex', '?')
    mod = getattr(d, 'Modality',   '?')
    ax.set_title(
        f'ID:{pid}  Age:{age}  {sex}  {mod}\n'
        f'HU [{h.min():.0f}, {h.max():.0f}]  '
        f'unique:{len(np.unique(h))}',
        color=TITLE, fontsize=7.5
    )
    ax.axis('off')

for ax in axes.flat[n_show:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

---
## 1️⃣3️⃣  Dataset-Wide Pixel Statistics

In [ ]:
print(f'Computing pixel statistics for all {len(dcm_files)} files…')
stats = []
for path in dcm_files:
    d = dcmread(path)
    h = to_hu(d)
    stats.append({
        'file'    : os.path.basename(path),
        'mean_hu' : float(h.mean()),
        'std_hu'  : float(h.std()),
        'min_hu'  : float(h.min()),
        'max_hu'  : float(h.max()),
        'unique'  : int(len(np.unique(h))),
        'Age'     : parse_age(getattr(d, 'PatientAge', None)),
        'Sex'     : getattr(d, 'PatientSex', '?'),
        'Modality': getattr(d, 'Modality',   'CT'),
    })

stats_df = pd.DataFrame(stats)
print(f'✓ Done')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('Pixel Statistics Across All CT Slices',
             color='white', fontsize=13, fontweight='bold')

def style(ax, title, xlabel, ylabel='Count'):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=TITLE, fontsize=11)
    ax.set_xlabel(xlabel, color=LABEL)
    ax.set_ylabel(ylabel, color=LABEL)
    ax.tick_params(colors=LABEL)
    for s in ax.spines.values(): s.set_edgecolor(BORDER)

# Mean HU per slice
axes[0].hist(stats_df['mean_hu'], bins=30, color=BLUE, edgecolor=BG)
style(axes[0], 'Mean HU per Slice', 'Mean HU')

# Mean vs Std HU scatter coloured by modality
pal = dict(zip(stats_df['Modality'].unique(), [BLUE, GREEN, PINK, YELLOW]))
for mod, grp in stats_df.groupby('Modality'):
    axes[1].scatter(grp['mean_hu'], grp['std_hu'],
                    c=pal.get(mod, BLUE), alpha=0.7, s=25,
                    edgecolors='none', label=mod)
axes[1].legend(labelcolor='white', facecolor='#21262d',
               edgecolor=BORDER, fontsize=9)
style(axes[1], 'Mean HU vs Std HU', 'Mean HU', 'Std HU')

# Unique HU values per slice vs 8-bit limit
axes[2].hist(stats_df['unique'], bins=30, color=YELLOW, edgecolor=BG)
axes[2].axvline(256, color=PINK, lw=1.5, ls='--', label='8-bit limit (256)')
axes[2].legend(labelcolor='white', facecolor='#21262d',
               edgecolor=BORDER, fontsize=9)
style(axes[2], 'Unique HU Values per Slice', 'Unique Values')

plt.tight_layout()
plt.show()

---
## 1️⃣4️⃣  Export — Save as PNG or Lossless NPY

In [ ]:
def export_dicom_files(file_list, out_dir, fmt='png',
                       wc=60, ww=400, max_files=None):
    """
    Export DICOM files to PNG (display) or NPY (lossless HU).

    Parameters
    ----------
    file_list : list of .dcm paths
    out_dir   : output directory
    fmt       : 'png' — 8-bit windowed (display / annotation)
                'npy' — float32 HU array (lossless, recommended for ML)
    wc, ww    : window centre / width (only for fmt='png')
    max_files : limit number of files
    """
    os.makedirs(out_dir, exist_ok=True)
    targets = file_list[:max_files] if max_files else file_list

    for path in targets:
        stem = os.path.splitext(os.path.basename(path))[0]
        try:
            d = dcmread(path)
            h = to_hu(d)
            if fmt == 'png':
                Image.fromarray(apply_window(h, wc, ww)).save(
                    os.path.join(out_dir, stem + '.png'))
            elif fmt == 'npy':
                np.save(os.path.join(out_dir, stem + '.npy'),
                        h.astype(np.float32))
        except Exception as e:
            print(f'  skip {stem}: {e}')

    n = len(glob.glob(os.path.join(out_dir, f'*.{fmt}')))
    print(f'✓ Exported {n} {fmt.upper()} files → {out_dir}')


# Uncomment to run:
# export_dicom_files(dcm_files, './exports_png', fmt='png')
# export_dicom_files(dcm_files, './exports_npy', fmt='npy')

print('export_dicom_files() ready.')
print('  fmt="npy" → lossless float32 HU  (recommended for ML)')
print('  fmt="png" → 8-bit windowed image  (display / labelling only)')

---
## ✅  Summary

| Step | What was done |
|---|---|
| **Load** | Read `.dcm` files from `dicom_dir/` with `pydicom` |
| **All tags** | Printed every DICOM tag (patient, study, acquisition, image) |
| **Metadata table** | Extracted tags for all files → pandas DataFrame |
| **Pixel data** | Converted raw values → Hounsfield Units |
| **Windowing** | 6 clinical presets — Brain, Subdural, Stroke, Bone, Lung, Abdomen |
| **Comparison** | Showed information loss when converting to 8-bit JPEG |
| **Dashboard** | Edge map, HU profile, tissue segmentation, tag panel |
| **Batch stats** | Mean/std HU across all slices, modality comparison |
| **Export** | PNG (display) or NPY (lossless HU) for downstream pipelines |

### References
- [gpreda's original notebook](https://www.kaggle.com/code/gpreda/visualize-ct-dicom-data)
- [kmader/siim-medical-images](https://www.kaggle.com/datasets/kmader/siim-medical-images)
- [pydicom documentation](https://pydicom.github.io/pydicom/stable/)
- [DICOM standard](https://www.dicomstandard.org/)